In [2]:
!git clone https://github.com/davallejo/pronaca-mini-dwh.git
%cd pronaca-mini-dwh

Cloning into 'pronaca-mini-dwh'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 17.99 KiB | 837.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/pronaca-mini-dwh


In [3]:
!pip -q install pandas numpy scipy scikit-learn pyspark

In [4]:
!python src/generar_datos.py
!python src/etl.py

✅ Datos generados en data/raw/ (productos.csv, clientes.json, ventas.csv)
✅ Base creada: data/pronaca_dwh.db
   Ventas cargadas:    295
   Ventas rechazadas:  5

📊 Reporte de calidad (DAMA):
        dimension_dama                                      regla  registros_total  registros_error  porcentaje_ok
          Consistencia Ciudades con formato único (strip + title)               20                3          85.00
              Unicidad              Ventas sin duplicados exactos              302                2          99.34
          Consistencia       Fechas parseables (ISO y DD/MM/AAAA)              300                0         100.00
           Completitud                         id_cliente no nulo              300                1          99.67
               Validez                               cantidad > 0              300                2          99.33
Integridad referencial         id_producto existe en dim_producto              300                1          99.67
Inte

In [5]:
import sqlite3, pandas as pd
con = sqlite3.connect("data/pronaca_dwh.db")

sql = open("sql/02_consultas_analiticas.sql", encoding="utf-8").read()
for i, q in enumerate([x for x in sql.split(";") if "SELECT" in x], start=1):
    print(f"--- Consulta {i} ---")
    display(pd.read_sql(q, con))

--- Consulta 1 ---


,categoria,ventas_usd
0,Embutidos,35929.70
1,Pollo,35126.89
2,Cerdo,22442.93


--- Consulta 2 ---


,nombre,unidades,ventas_usd
0,Jamón cocido,1685,13387.20
1,Costilla de cerdo,1770,11434.84
2,Pechuga de pollo,2256,11369.55
3,Chuleta de cerdo,1897,11008.09
4,Nuggets de pollo,1628,10633.12


--- Consulta 3 ---


,canal,ciudad,ventas_usd
0,Distribuidor,Cuenca,10792.47
1,Tienda de barrio,Manta,9411.30
2,Supermercado,Manta,9119.98
3,Supermercado,Quito,8576.83
4,Restaurante,Guayaquil,7984.52
5,Distribuidor,Guayaquil,7034.04
6,Supermercado,Guayaquil,6448.62
7,Supermercado,Ambato,6189.96
8,Tienda de barrio,Cuenca,6103.49
9,Distribuidor,Quito,5726.69


--- Consulta 4 ---


,anio,mes,ventas_usd
0,2025,1,16109.65
1,2025,2,10520.77
2,2025,3,11562.21
3,2025,4,21500.05
4,2025,5,18276.17
5,2025,6,15530.67


In [6]:
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

m = pd.read_sql("""SELECT mes, SUM(ventas_usd) AS ventas
                   FROM v_ventas_mensuales GROUP BY mes ORDER BY mes""", con)

X = m[["mes"]].values
y = m["ventas"].values

modelo = LinearRegression().fit(X, y)
proximo = int(m["mes"].max()) + 1
print(f"Pronóstico mes {proximo}: ${modelo.predict([[proximo]])[0]:,.2f}")

r, p = stats.pearsonr(m["mes"], m["ventas"])
print(f"Correlación mes vs ventas: r={r:.2f} (p={p:.3f})")
print(f"Promedio: {np.mean(y):,.2f} | Desv. estándar: {np.std(y):,.2f}")

Pronóstico mes 7: $18,614.17
Correlación mes vs ventas: r=0.39 (p=0.439)
Promedio: 15,583.25 | Desv. estándar: 3,749.17


In [7]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("PronacaDWH").master("local[*]").getOrCreate()

fact = spark.read.csv("data/processed/fact_ventas.csv", header=True, inferSchema=True)
prod = spark.read.csv("data/processed/dim_producto.csv", header=True, inferSchema=True)

(fact.join(prod, "id_producto")
     .groupBy("categoria")
     .agg(F.round(F.sum("total"), 2).alias("ventas_usd"),
          F.sum("cantidad").alias("unidades"))
     .orderBy(F.desc("ventas_usd"))
     .show())

+---------+----------+--------+
|categoria|ventas_usd|unidades|
+---------+----------+--------+
|Embutidos|   35929.7|    7105|
|    Pollo|  35126.89|    7298|
|    Cerdo|  22442.93|    3667|
+---------+----------+--------+



In [8]:
!zip -rq resultados.zip data
from google.colab import files
files.download("resultados.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>